# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook walks through loading, overviewing, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Dataset Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id` values.

**Note:** In Croissant datasets, each RecordSet, Field, and Column has a unique `@id`.

In [ ]:
# Access record sets from metadata - all entities by @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}, Name: {rs.get('name', '')}")
        fields = rs.get('fields', [])
        for field in fields:
            print(f"  Field @id: {field['@id']}, Name: {field.get('name', '')}, DataType: {field.get('dataType', '')}")

### Example: Print Sample Records by RecordSet ID
We can use the `mlcroissant.Dataset.records()` method and specify a RecordSet `@id` to iterate over sample records. Make sure to use the correct record set `@id` obtained above.

In [ ]:
# List all record_set @id's for extraction
available_record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Available RecordSet @id's:", available_record_sets)

# Print a preview from the first available record set
if available_record_sets:
    preview_record_set_id = available_record_sets[0]
    for i, record in enumerate(dataset.records(record_set=preview_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record sets to preview.')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using their `@id` properties.

In [ ]:
# Extract all available record sets into DataFrames
dfs = {}
for rs_id in available_record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df

# List columns from the first record set
main_rs_id = available_record_sets[0] if available_record_sets else None
if main_rs_id:
    print(f"Columns in RecordSet {main_rs_id}:\n", dfs[main_rs_id].columns.tolist())
    dfs[main_rs_id].head()
else:
    print("No RecordSet dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Reference all columns and fields by their `@id` where possible.

In [ ]:
# Let's select a numeric field for analysis. We'll search for a likely score column (e.g., GAD-7 score column).
df = dfs[main_rs_id]

# Find numeric columns by inspecting column names
numeric_field_candidates = [col for col in df.columns if (
    'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower() or 'age' in col.lower())]

print("Numeric field candidates:", numeric_field_candidates)

# Select first candidate as numeric_field_id
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else None

# Apply threshold filtering if numeric_field is available
if numeric_field_id:
    # Simple threshold: show rows with GAD-7 score > 10
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field (e.g., 'gender', 'age', 'education')
    group_candidates = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower())]
    group_field = group_candidates[0] if group_candidates else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the key numeric field (e.g., GAD-7 score) and, if available, group by a demographic variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarized below are some of the key findings and observations from exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya:

- The dataset provides rich survey records on mental health, with demographic details and psychological assessment scores.
- Numeric scores such as GAD-7 can be extracted and analyzed for population-level trends and potential high-risk groups.
- Grouping and visualizing scores by demographic variables (e.g., gender, education) reveals patterns valuable for public health and research.

**Next steps** may include modeling risk factors, mapping spatial distributions, or integrating additional survey rounds or populations.